# NCDD PDF downloader

## 1. Install packages

In [ ]:
%pip install -q requests beautifulsoup4 pymupdf pandas
%pip install -q ipykernel

## 2. Configuration

Change the values in this cell when you want to scrape another category or limit the test.

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import re
import time

import fitz
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://library.ncdd.gov.kh"
CATEGORY_ID = 7
PER_PAGE = 12
START_PAGE = 1
MAX_PAGES = 1          # Number of pages; use None until no data is found
START_URL = f"{BASE_URL}/product?category={CATEGORY_ID}&page={START_PAGE}&perpage={PER_PAGE}"
DELAY_SECONDS = 2
TIMEOUT_SECONDS = 30
DOWNLOAD_ATTEMPTS = 3
RETRY_WAIT_SECONDS = 5
HEADERS = {"User-Agent": "LegalRAG academic PDF downloader (contact: researcher)"}
KHMER_DIGITS = str.maketrans("០១២៣៤៥៦៧៨៩", "0123456789")
HTML_TAGS = {
    "document_cards": "div.product-card",
    "document_links": 'a[href*="/detail/"]',
    "document_title": ".product-card__name a",
    "feature_items": "ul.product-card__features-list > li",
    "category_badge": ".product-card__badge",
    "cover_image": "img.product-image__img",
}

OUTPUT_DIR = Path("data/source")
CHECKPOINT = Path("data/documents.csv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

## 3. Inspect the HTML tags

These selectors are the hooks used to find documents and PDF links. The cell also prints real matching tags from the website.

In [ ]:
def get_soup(url):
    response = requests.get(url, headers=HEADERS, timeout=TIMEOUT_SECONDS)
    if response.status_code in (401, 403):
        raise PermissionError(f"Access denied: HTTP {response.status_code}")
    if response.status_code == 429:
        raise RuntimeError("Too many requests (429). Stop and try again later.")
    response.raise_for_status()
    return response, BeautifulSoup(response.text, "html.parser")


## 4. Extract documents and metadata from listing pages

Each product card contributes its title, category, total page count, publication year, topic, views, cover URL, and raw metadata.

In [ ]:
def number_from_text(value):
    normalized = value.translate(KHMER_DIGITS)
    match = re.search(r"\d[\d,]*", normalized)
    return int(match.group().replace(",", "")) if match else None


def extract_documents(soup, page_url):
    documents = {}

    for card in soup.select(HTML_TAGS["document_cards"]):
        link = card.select_one(HTML_TAGS["document_links"])
        if not link:
            continue

        detail_url = urljoin(page_url, link.get("href", ""))
        parts = urlparse(detail_url).path.rstrip("/").split("/")

        if len(parts) < 2 or parts[-2] != "detail" or not parts[-1].isdigit():
            continue

        document_id = parts[-1]

        features = {}
        for item in card.select(HTML_TAGS["feature_items"]):
            label, separator, value = item.get_text(" ", strip=True).partition(":")
            if separator:
                features[label.strip()] = value.strip()

        category = card.select_one(HTML_TAGS["category_badge"])
        title = card.select_one(HTML_TAGS["document_title"])
        cover = card.select_one(HTML_TAGS["cover_image"])

        documents[document_id] = {
            "document_id": document_id,
            "title": title.get_text(" ", strip=True) if title else "",
            "detail_url": detail_url,
            "category": (
                category.get_text(" ", strip=True)
                if category
                else features.get("ប្រភេទ", "")
            ),
            "total_pages": number_from_text(features.get("ចំនួនទំព័រ", "")),
            "publication_year": number_from_text(features.get("ឆ្នាំដាក់ចេញ", "")),
            "topic": features.get("ប្រធានបទ", ""),
            "cover_url": urljoin(page_url, cover.get("src", "")) if cover else "",
        }

    return list(documents.values())


def discover_documents():
    documents = {}
    page_number = START_PAGE
    pages_read = 0
    while MAX_PAGES is None or pages_read < MAX_PAGES:
        url = f"{BASE_URL}/product?category={CATEGORY_ID}&page={page_number}&perpage={PER_PAGE}"
        print("Reading:", url)
        _, soup = get_soup(url)
        page_documents = extract_documents(soup, url)
        if not page_documents:
            print(f"No documents found on page {page_number}; stopping.")
            break
        for document in page_documents:
            documents[document["document_id"]] = document
        print(f"Found {len(page_documents)} documents")
        pages_read += 1
        page_number += 1
        time.sleep(DELAY_SECONDS)
    return list(documents.values())


## 5. Define the direct PDF URL step

This function adds `/download` to each detail URL. It only defines the step; the final cell runs it.

In [ ]:
def add_pdf_urls(documents):
    for document in documents:
        detail_url = document["detail_url"].rstrip("/")
        document["pdf_url"] = f"{detail_url}/download"
    return documents

def save_documents(documents):
    dataframe = pd.DataFrame(documents)
    dataframe.to_csv(CHECKPOINT, index=False, encoding="utf-8-sig")
    return dataframe

## 6. Define the PDF downloader

This function validates the PDF and retries failed downloads three times. It only defines the function; the final cell runs it.

In [ ]:
def download_pdf(pdf_url, output_path, referer):
    last_error = None
    temporary_path = output_path.with_suffix(output_path.suffix + ".part")
    for attempt in range(1, DOWNLOAD_ATTEMPTS + 1):
        try:
            response = requests.get(
                pdf_url,
                headers={**HEADERS, "Referer": referer},
                timeout=TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            if "pdf" not in response.headers.get("Content-Type", "").lower():
                raise ValueError("The URL did not return a PDF")
            if not response.content.startswith(b"%PDF-"):
                raise ValueError("The downloaded file is not a valid PDF")
            temporary_path.write_bytes(response.content)
            with fitz.open(temporary_path) as pdf:
                if pdf.page_count == 0:
                    raise ValueError("The PDF has no pages")
            temporary_path.replace(output_path)
            return output_path
        except Exception as error:
            last_error = error
            temporary_path.unlink(missing_ok=True)
            print(f"Download attempt {attempt}/{DOWNLOAD_ATTEMPTS} failed: {error}")
            if attempt < DOWNLOAD_ATTEMPTS:
                time.sleep(RETRY_WAIT_SECONDS)
    raise RuntimeError(f"Download failed after {DOWNLOAD_ATTEMPTS} attempts: {last_error}")

def download_documents(documents):
    downloadable = [document for document in documents if document.get("pdf_url")]
    for document in documents:
        document.setdefault("local_pdf_path", "")
        document.setdefault("download_status", "")
        document.setdefault("download_error", "")

    for number, document in enumerate(downloadable, start=1):
        output_path = OUTPUT_DIR / f"{document['document_id']}.pdf"
        try:
            if output_path.exists() and output_path.read_bytes()[:5] == b"%PDF-":
                status = "already_exists"
            else:
                download_pdf(document["pdf_url"], output_path, document["detail_url"])
                status = "downloaded"
            document["local_pdf_path"] = str(output_path)
            document["download_status"] = status
            document["download_error"] = ""
            print(f"[{number}/{len(downloadable)}] {document['document_id']}: {status}")
        except Exception as error:
            document["download_status"] = "failed"
            document["download_error"] = str(error)
            print(f"[{number}/{len(downloadable)}] {document['document_id']}: failed — {error}")
        time.sleep(DELAY_SECONDS)
    return documents

## 7. Run the complete pipeline

This final cell runs all previously defined functions: discover metadata, create direct PDF URLs, save `documents.csv`, download PDFs, and update the same CSV.

In [ ]:
listing_response, listing_soup = get_soup(START_URL)
document_tags = listing_soup.select(HTML_TAGS["document_links"])

documents = add_pdf_urls(discover_documents())
documents_df = save_documents(documents)

documents = download_documents(documents)
documents_df = save_documents(documents)
display(documents_df["download_status"].value_counts().to_frame("count"))